# AI Morph Ads — Colab / Kaggle runner

One product image → four ad creatives (Luxury / Minimal / Urban / Nostalgic).

**Runtime:** GPU (T4 is enough — 15 GB VRAM).

This notebook:
1. Clones ComfyUI + required custom nodes
2. Clones *this* project and installs Python deps
3. Downloads all model weights (base models + 4 style LoRAs) into `ComfyUI/models/...`
4. Starts ComfyUI as a background server
5. Runs `scripts/run_pipeline.py` for a product image you upload

> **LoRAs** are now auto-downloaded from HuggingFace by `scripts/download_models.py`. If you want to override them with your own `.safetensors` from CivitAI, either run the download step with `--skip-loras` and drop your files manually into `ComfyUI/models/loras/`, or edit `configs/variants.yaml` to point at different filenames.

## 1. Verify GPU

In [1]:
!nvidia-smi

Thu Apr 23 14:07:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone ComfyUI + custom nodes + this project

In [2]:
import os, sys, subprocess, pathlib

WORK = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path('/kaggle/working')
os.chdir(WORK)
print('Working dir:', WORK)

if not (WORK / 'ComfyUI').exists():
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git

NODES = WORK / 'ComfyUI' / 'custom_nodes'
NODES.mkdir(parents=True, exist_ok=True)

for name, repo in [
    ('ComfyUI-Manager',           'https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('ComfyUI_IPAdapter_plus',    'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    ('comfyui_controlnet_aux',    'https://github.com/Fannovel16/comfyui_controlnet_aux.git'),
]:
    target = NODES / name
    if not target.exists():
        print(f'git clone {name}')
        subprocess.run(['git', 'clone', '--depth', '1', repo, str(target)], check=True)

print('\nInstalling ComfyUI requirements...')
!pip install -q -r ComfyUI/requirements.txt

print('Installing custom-node requirements...')
for name in ['ComfyUI-Manager', 'ComfyUI_IPAdapter_plus', 'comfyui_controlnet_aux']:
    req = NODES / name / 'requirements.txt'
    if req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)

Working dir: /content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 915, done.
remote: Counting objects: 100% (915/915), done.
remote: Compressing objects: 100% (799/799), done.
remote: Total 915 (delta 83), reused 833 (delta 77), pack-reused 0 (from 0)
Receiving objects: 100% (915/915), 10.50 MiB | 15.98 MiB/s, done.
Resolving deltas: 100% (83/83), done.
git clone ComfyUI-Manager
git clone ComfyUI_IPAdapter_plus
git clone comfyui_controlnet_aux

Installing ComfyUI requirements...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 94.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 124.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 MB 9.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 13.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.9 MB/s e

In [6]:
!git clone https://github.com/seemyoon/AI_Morph_Ads_with_ComfyUI.git

Cloning into 'AI_Morph_Ads_with_ComfyUI'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 23 (delta 2), reused 23 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 1.45 MiB | 4.66 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [7]:
PROJECT = WORK / 'AI_Morph_Ads_with_ComfyUI'
assert PROJECT.exists(), f'Upload / clone the project to {PROJECT}'

!pip install -q -r "$PROJECT/requirements.txt"

In [10]:
print('WORK: ', WORK)
print("PROJECT: ",PROJECT)

WORK:  /content
PROJECT:  /content/AI_Morph_Ads_with_ComfyUI


## 3. Download model weights

~10 GB total with SDXL. First run takes 5–10 min on a Colab T4; subsequent sessions skip anything already on disk.

Optional but recommended: set `HF_TOKEN` in your Colab environment before this step to avoid HuggingFace throttling and increase download rate limits.

In [16]:
!python "$PROJECT/scripts/download_models.py" --comfy-root "$WORK/ComfyUI"

Target: /content/ComfyUI/models

--> checkpoints/v1-5-pruned-emaonly.safetensors
  [skip] v1-5-pruned-emaonly.safetensors already present
--> vae/vae-ft-mse-840000-ema-pruned.safetensors
  [skip] vae-ft-mse-840000-ema-pruned.safetensors already present
--> controlnet/control_v11p_sd15_canny.pth
  [skip] control_v11p_sd15_canny.pth already present
--> controlnet/control_v11f1p_sd15_depth.pth
  [skip] control_v11f1p_sd15_depth.pth already present
--> ipadapter/ip-adapter-plus_sd15.safetensors
  [skip] ip-adapter-plus_sd15.safetensors already present
--> clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors
  [skip] CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors already present
--> checkpoints/sd_xl_base_1.0.safetensors
  [skip] sd_xl_base_1.0.safetensors already present

LoRA files — manual download required (CivitAI)
Place the following 4 files into:  /content/ComfyUI/models/loras

  - epiCRealism_Luxury.safetensors
  - minimalist_studio_v2.safetensors
  - neon_street_night.safetensors
 

In [17]:
from pathlib import Path

files = list(Path('/content/ComfyUI/models').iterdir())
print([p.name for p in files])

['style_models', 'clip_vision', 'vae', 'checkpoints', 'latent_upscale_models', 'controlnet', 'clip', 'loras', 'diffusers', 'hypernetworks', 'embeddings', 'diffusion_models', 'ipadapter', 'unet', 'audio_encoders', 'configs', 'frame_interpolation', 'gligen', 'model_patches', 'upscale_models', 'text_encoders', 'photomaker', 'vae_approx']


In [19]:
from pathlib import Path

loras = list(Path('/content/ComfyUI/models/loras').iterdir())
print([p.name for p in loras])

['put_loras_here']


## 4. Verify auto-downloaded LoRAs + upload product image

By default, `scripts/download_models.py` now auto-downloads the 4 style LoRAs from HuggingFace into `ComfyUI/models/loras/` and creates the expected alias for the Minimal variant.

Use the Colab file browser (left sidebar) to upload only:
- 1 product image (PNG/JPG) into `AI Morph Ads/inputs/`

Optional:
- If you want custom LoRAs, rerun the download step with `--skip-loras` and place your own `.safetensors` under `ComfyUI/models/loras/` (filenames must match `configs/variants.yaml`).
- Set `HF_TOKEN` in Colab environment variables to avoid HuggingFace throttling.

In [15]:
expected_loras = {
    'epiCRealism_Luxury.safetensors',
    'minimalist_studio_v2.safetensors',
    'neon_street_night.safetensors',
    'film_kodak_portra_v3.safetensors',
}

loras = sorted((WORK / 'ComfyUI' / 'models' / 'loras').glob('*.safetensors'))
found_loras = {p.name for p in loras}

print('LoRAs found:')
for p in loras:
    print(' -', p.name)

missing = sorted(expected_loras - found_loras)
if missing:
    print('\n[WARN] Missing expected LoRA files:')
    for name in missing:
        print(' -', name)
    print('If HF download was rate-limited, set HF_TOKEN and rerun the download cell.')
else:
    print('\n[OK] All expected LoRA files are present.')

inputs = sorted((PROJECT / 'inputs').glob('*'))
print('\nProduct images found:')
for p in inputs:
    if p.is_file():
        print(' -', p.name)

LoRAs found:

Product images found:
 - .gitkeep


## 5. Start ComfyUI as a background server

We launch it with `--listen 127.0.0.1 --port 8188` and `--medvram` so it fits inside T4's 15 GB budget (SDXL base is loaded in stage B only).

In [ ]:
import subprocess, time, requests

LOG = WORK / 'comfyui.log'
log_f = open(LOG, 'w')
proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--medvram'],
    cwd=str(WORK / 'ComfyUI'),
    stdout=log_f, stderr=subprocess.STDOUT,
)
print('ComfyUI PID:', proc.pid)

for i in range(60):
    try:
        r = requests.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if r.ok:
            print('ComfyUI is up.')
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError(f'ComfyUI did not start in time — check {LOG}')

## 6. Run the 4-variant pipeline

In [ ]:
PRODUCT_FILENAME = 'bottle.png'
BRAND            = 'NOIR'
TAGLINE          = 'Timeless.'
PRODUCT_NAME     = 'perfume bottle'

!python "$PROJECT/scripts/run_pipeline.py" \
    --product "$PROJECT/inputs/$PRODUCT_FILENAME" \
    --brand "$BRAND" \
    --tagline "$TAGLINE" \
    --product-name "$PRODUCT_NAME" \
    --server http://127.0.0.1:8188

## 7. Preview the 4 creatives

In [ ]:
from IPython.display import Image, display
for p in sorted((PROJECT / 'outputs').glob('*.png')):
    if p.name.endswith('_raw.png'):
        continue
    print(p.name)
    display(Image(str(p), width=512))